In [ ]:
# Load standard Python packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import ols
from IPython.display import HTML
import statsmodels.api as sm

## EDA1: shape, dtypes, stats, etc.

In [ ]:
import pandas as pd
import numpy as np


# read in datasets
acled = pd.read_csv('https://raw.githubusercontent.com/lukef533/Russo-Ukrainian-Airstrike-Analysis/refs/heads/main/DATA/ACLED%20Data_2026-02-13.csv')
missile_attacks_daily = pd.read_csv('https://raw.githubusercontent.com/lukef533/Russo-Ukrainian-Airstrike-Analysis/refs/heads/main/DATA/missile_attacks_daily.csv')
missiles_uavs = pd.read_csv('https://raw.githubusercontent.com/lukef533/Russo-Ukrainian-Airstrike-Analysis/refs/heads/main/DATA/missiles_and_uavs.csv')
missile_combined = pd.read_csv('https://raw.githubusercontent.com/lukef533/Russo-Ukrainian-Airstrike-Analysis/refs/heads/main/DATA/combined.csv')

In [ ]:
print("Shape of 'acled' DataFrame:")
display(acled.shape)
print("\nSummary statistics for 'acled' DataFrame:")
display(acled.describe())
print("\nInfo for 'acled' DataFrame:")
display(acled.info())

print("\n\nShape of 'missle_attacks_daily' DataFrame:")
display(missile_attacks_daily.shape)
print("\nSummary statistics for 'missle_attacks_daily' DataFrame:")
display(missile_attacks_daily.describe())
print("\nInfo for 'missle_attacks_daily' DataFrame:")
display(missile_attacks_daily.info())

print("\n\nShape of 'missles_uavs' DataFrame:")
display(missiles_uavs.shape)
print("\nSummary statistics for 'missles_uavs' DataFrame:")
display(missiles_uavs.describe())
print("\nInfo for 'missles_uavs' DataFrame:")
display(missiles_uavs.info())

In [ ]:
# # merge dailys and missile/uav specs

# missile_combined = missile_attacks_daily.merge(
#     missiles_uavs,
#     on='model',
#     how='left'
# )

In [ ]:
#missile_combined.to_csv('combined.csv')

In [ ]:
print(missile_attacks_daily.shape)
print(missiles_uavs.shape)
print(missile_combined.shape)

In [ ]:
# Convert dates to datetime
acled['event_date'] = pd.to_datetime(acled['event_date'])

missile_combined['attack_date'] = missile_combined['time_start'].str.split(' ').str[0]
missile_combined['attack_date'] = pd.to_datetime(missile_combined['attack_date'])


In [ ]:
print(acled.dtypes)

In [ ]:
import pandas as pd

print("Timestamp column info:")
print("Data type:", acled['timestamp'].dtype)
print("\nFirst 10 values:")
print(acled['timestamp'].head(10))

print("\nBasic stats:")
print("Min:", acled['timestamp'].min())
print("Max:", acled['timestamp'].max())
print("Unique values:", acled['timestamp'].nunique())

print("\nSample rows with timestamp, event_date, and event_type:")
print(acled[['event_date', 'timestamp', 'event_type', 'location']].head(20))

In [ ]:
print(acled.event_date.nunique())
print(acled.shape)

In [ ]:
print(missile_combined.attack_date.nunique())
print(missile_combined.shape)

In [ ]:
acled.fatalities.value_counts(normalize = True)

In [ ]:
# Check the format of time_start
print("Sample time_start values:")
print(missile_combined['time_start'].head(20))

# Count how many have time (contain a space or colon)
has_time = missile_combined['time_start'].str.contains(' ', na=False) | missile_combined['time_start'].str.contains(':', na=False)
just_date = ~has_time

print("\n=== TIME FORMAT BREAKDOWN ===")
print(f"Date only (no time): {just_date.sum()} rows")
print(f"Date and time: {has_time.sum()} rows")
print(f"Total: {len(missile_combined)} rows")

print(f"\nPercentage:")
print(f"Date only: {100*just_date.sum()/len(missile_combined):.1f}%")
print(f"Date and time: {100*has_time.sum()/len(missile_combined):.1f}%")

# Show examples of each
print("\n=== EXAMPLES: DATE ONLY ===")
print(missile_combined[just_date]['time_start'].head(5))

print("\n=== EXAMPLES: DATE AND TIME ===")
print(missile_combined[has_time]['time_start'].head(5))


In [ ]:
missile_combined.category.value_counts(normalize = True)

combined_uav_only = missile_combined[missile_combined.category == 'UAV']



# Check the format of time_start
print("Sample time_start values:")
print(combined_uav_only['time_start'].head(20))

# Count how many have time (contain a space or colon)
has_time = combined_uav_only['time_start'].str.contains(' ', na=False) | combined_uav_only['time_start'].str.contains(':', na=False)
just_date = ~has_time

print("\n=== TIME FORMAT BREAKDOWN ===")
print(f"Date only (no time): {just_date.sum()} rows")
print(f"Date and time: {has_time.sum()} rows")
print(f"Total: {len(combined_uav_only)} rows")

print(f"\nPercentage:")
print(f"Date only: {100*just_date.sum()/len(combined_uav_only):.1f}%")
print(f"Date and time: {100*has_time.sum()/len(combined_uav_only):.1f}%")

# Show examples of each
print("\n=== EXAMPLES: DATE ONLY ===")
print(combined_uav_only[just_date]['time_start'].head(5))

print("\n=== EXAMPLES: DATE AND TIME ===")
print(combined_uav_only[has_time]['time_start'].head(5))


In [ ]:
missile_combined.target.value_counts()

In [ ]:
acled.fatalities.mean()

## EDA2: Graphs

In [ ]:


# Convert dates
acled['event_date'] = pd.to_datetime(acled['event_date'])
missile_combined['time_start'] = pd.to_datetime(missile_combined['time_start'], format='mixed', errors='coerce')
missile_combined['attack_date'] = missile_combined['time_start'].dt.date

# ============ VIZ 1: Event Types in ACLED ============
fig, ax = plt.subplots(figsize=(10, 6))
event_counts = acled['event_type'].value_counts().head(10)
event_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Events')
ax.set_title('Top 10 Event Types in Ukraine Conflict (ACLED)')
plt.tight_layout()
plt.savefig('viz1_event_types.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ VIZ 2: Missile/UAV Types ============
fig, ax = plt.subplots(figsize=(10, 6))
missile_counts = missile_combined['model'].value_counts().head(10)
missile_counts.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Number of Attacks')
ax.set_title('Top 10 Missile/UAV Types Used')
plt.tight_layout()
plt.savefig('viz2_missile_types.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ VIZ 3: Events Over Time ============
fig, ax = plt.subplots(figsize=(12, 6))
events_by_date = acled.groupby(acled['event_date'].dt.to_period('M')).size()
events_by_date.plot(ax=ax, color='darkblue', linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Events')
ax.set_title('Conflict Events Over Time')
plt.tight_layout()
plt.savefig('viz3_events_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ VIZ 4: Missile Attacks Over Time ============
fig, ax = plt.subplots(figsize=(12, 6))
missile_combined['attack_date'] = pd.to_datetime(missile_combined['attack_date'])
missiles_by_date = missile_combined.groupby(missile_combined['attack_date'].dt.to_period('M'))['launched'].sum()
missiles_by_date.plot(ax=ax, color='darkred', linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Total Missiles/UAVs Launched')
ax.set_title('Missile/UAV Launches Over Time')
plt.tight_layout()
plt.savefig('viz4_missiles_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ VIZ 5: Top Affected Regions (ACLED) ============
fig, ax = plt.subplots(figsize=(10, 6))
region_counts = acled['admin1'].value_counts().head(10)
region_counts.plot(kind='barh', ax=ax, color='green')
ax.set_xlabel('Number of Events')
ax.set_title('Top 10 Most Affected Regions')
plt.tight_layout()
plt.savefig('viz5_regions.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ VIZ 6: Fatalities Over Time ============
fig, ax = plt.subplots(figsize=(12, 6))
fatalities_by_date = acled.groupby(acled['event_date'].dt.to_period('M'))['fatalities'].sum()
fatalities_by_date.plot(ax=ax, color='purple', linewidth=2, marker='o')
ax.set_xlabel('Month')
ax.set_ylabel('Total Fatalities')
ax.set_title('Monthly Fatalities from Conflict Events')
plt.tight_layout()
plt.savefig('viz6_fatalities_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ VIZ 7: Missile Success Rate ============
fig, ax = plt.subplots(figsize=(10, 6))
missile_combined['success_rate'] = (missile_combined['destroyed'] / missile_combined['launched']) * 100
success_by_model = missile_combined.groupby('model')['success_rate'].mean().sort_values(ascending=False).head(10)
success_by_model.plot(kind='barh', ax=ax, color='orange')
ax.set_xlabel('Success Rate (%)')
ax.set_title('Missile Destruction Rate by Type (Top 10)')
plt.tight_layout()
plt.savefig('viz7_success_rate.png', dpi=150, bbox_inches='tight')
plt.show()

# ============ SUMMARY STATS ============
print("=== SUMMARY STATISTICS ===\n")
print(f"Total ACLED Events: {len(acled)}")
print(f"Date Range: {acled['event_date'].min()} to {acled['event_date'].max()}")
print(f"Total Fatalities: {acled['fatalities'].sum()}")
print(f"\nTotal Missile/UAV Records: {len(missile_combined)}")
print(f"Total Missiles Launched: {missile_combined['launched'].sum():.0f}")
print(f"Total Missiles Destroyed: {missile_combined['destroyed'].sum():.0f}")
print(f"Overall Destruction Rate: {(missile_combined['destroyed'].sum() / missile_combined['launched'].sum()) * 100:.1f}%")

# Geospatial Analysis

2D Map with ACLED events with fatalities.

In [ ]:
print(acled['civilian_targeting'].value_counts(dropna=False, normalize=True))

In [ ]:
import folium
import json
import requests # Import the requests library

# Filter acled DataFrame for events with fatalities > 0 (as per previous request)
filtered_acled = acled[acled['fatalities'] > 0].copy()

# Create a base map centered around Ukraine
m_color_coded = folium.Map(location=[49.0, 32.0], zoom_start=6, tiles="cartodb positron")



# Load Json file with Ukraine border coordinates
geojso_url = 'https://raw.githubusercontent.com/lukef533/Ukraine-Missle-Strikes-Data/refs/heads/main/ukraine.json'
response = requests.get(geojso_url)
country_geo = json.loads(response.text)


# Filter for a Ukraine
# and add it to the map with custom styling
folium.GeoJson(
    country_geo,
    name='Ukraine',
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 2,
        'dashArray': '5, 5'
    }
).add_to(m_color_coded)


# Add a marker for each event, color-coding by civilian targeting
for idx, row in filtered_acled.iterrows():
    if pd.isna(row['civilian_targeting']):
        color = 'darkblue'
        targeting_info = 'Non-Civilian Targeted'
    else:
        color = 'darkred'
        targeting_info = 'Civilian Targeted'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        tooltip=f"Type: {row['event_type']}<br>Location: {row['location']}<br>Fatalities: {row['fatalities']}<br>Targeting: {targeting_info}"
    ).add_to(m_color_coded)

# Display the map
display(m_color_coded)

## 3D Map

In [ ]:
pip install pydeck matplotlib

In [ ]:
pip install streamlit

In [ ]:
import pandas as pd
import geopandas as gpd
import pydeck as pdk
import matplotlib
import matplotlib.pyplot as plt

# 1. Aggregate Regional Data (Attacks and Fatalities)
# Calculate total fatalities and total attacks for each region (admin1)
regional_stats = acled.groupby('admin1').agg(
    attack_count=('event_id_cnty', 'count'),
    regional_fatalities=('fatalities', 'sum')
).reset_index()

# Normalize the attack count for the color intensity scale
min_attacks = regional_stats['attack_count'].min()
max_attacks = regional_stats['attack_count'].max()
regional_stats['normalized_intensity'] = (regional_stats['attack_count'] - min_attacks) / (max_attacks - min_attacks)

# 2. Load and Prepare Ukraine GeoJSON
geojso_url = "https://raw.githubusercontent.com/martynafford/natural-earth-geojson/master/10m/cultural/ne_10m_admin_1_states_provinces.json"
world_regions = gpd.read_file(geojson_url)

# FIX: Broaden filter to include Crimea/Sevastopol if they are labeled as 'Russia' or 'Disputed'
crimea_names = ['Crimea', 'Avtonomna Respublika Krym', 'Krym', 'Sevastopol', 'm. Sevastopol']
ukraine_regions = world_regions[
    (world_regions['admin'] == 'Ukraine') |
    (world_regions['name'].isin(crimea_names))
].copy()

# Robust Mapping to align GeoJSON names with ACLED
name_mapping = {
    'Kiev': 'Kyiv',
    'Kiev City': 'Kyiv City',
    "Dnipropetrovs'k": 'Dnipropetrovsk',
    'Zaporizhzhya': 'Zaporizhia',
    'Mykolayiv': 'Mykolaiv',
    'Avtonomna Respublika Krym': 'Crimea',
    'Krym': 'Crimea',
    'm. Sevastopol': 'Sevastopol',
    "Donets'k": 'Donetsk',
    'Odessa': 'Odesa',
    "Luhans'k": 'Luhansk',
    "Khmel'nyts'kyy": 'Khmelnytskyi',
    "Ivano-Frankivs'k": 'Ivano-Frankivsk',
    "L'viv": 'Lviv',
    "Ternopil'": 'Ternopil',
    "Vinnytsya": 'Vinnytsia'
}
ukraine_regions['name'] = ukraine_regions['name'].replace(name_mapping)

# Merge aggregated stats into the GeoJSON
ukraine_regions = ukraine_regions.merge(
    regional_stats,
    left_on='name',
    right_on='admin1',
    how='left'
)

# --- Tooltip Preparation: Unify Column Names to avoid "undefined" ---
ukraine_regions['attack_count'] = ukraine_regions['attack_count'].fillna(0)
ukraine_regions['regional_fatalities'] = ukraine_regions['regional_fatalities'].fillna(0)
ukraine_regions['individual_fatalities'] = "N/A (Hover over a bar)"
ukraine_regions['admin1_display'] = ukraine_regions['name']

# Color the regions based on attack intensity
cmap = matplotlib.colormaps['Oranges']
ukraine_regions['fill_color'] = ukraine_regions['normalized_intensity'].apply(
    lambda x: [int(c * 255) for c in cmap(x)[:3]] + [180] if not pd.isna(x) else [0, 0, 0, 0]
)

# 3. Prepare Strike Data (3D Bars)
fatal_strikes = acled[acled['fatalities'] > 0].copy()
fatal_strikes = fatal_strikes.dropna(subset=['longitude', 'latitude'])

# Merge regional stats into individual strike data for the tooltip
fatal_strikes = fatal_strikes.merge(regional_stats, on='admin1', how='left')
fatal_strikes['individual_fatalities'] = fatal_strikes['fatalities']
fatal_strikes['admin1_display'] = fatal_strikes['admin1']

# 4. Define PyDeck Layers
region_layer = pdk.Layer(
    "GeoJsonLayer",
    ukraine_regions,
    opacity=0.8,
    stroked=True,
    filled=True,
    get_fill_color="fill_color",
    get_line_color= [80,80,80],
    pickable=True
)

column_layer = pdk.Layer(
    "ColumnLayer",
    data=fatal_strikes,
    get_position=['longitude', 'latitude'],
    get_elevation='individual_fatalities',
    elevation_scale=3000,
    radius=3000,
    get_fill_color= [136, 8, 8], # Pure Red bars
    pickable=True,
    auto_highlight=True,
)

view_state = pdk.ViewState(latitude=49.0, longitude=32.0, zoom=5.5, pitch=50, bearing=0)

# Unified Tooltip to fix the "undefined" error
tooltip = {
    "html": """
        <b>Region:</b> {admin1_display} <br/>
        <b>Total Regional Attacks:</b> {attack_count} <br/>
        <b>Total Regional Fatalities:</b> {regional_fatalities} <br/>
        <hr style="margin: 5px 0;">
        <b>Strike Fatalities:</b> {individual_fatalities}
    """,
    "style": {"background": "#333333", "color": "white", "font-family": 'Arial', "z-index": "10000"}
}

r = pdk.Deck(layers=[region_layer, column_layer], initial_view_state=view_state, tooltip=tooltip, map_style='dark')

# Display the map
display(r.show())

# 5. Legend for the Poster
fig, ax = plt.subplots(figsize=(8, 1))
fig.subplots_adjust(bottom=0.5)
norm = matplotlib.colors.Normalize(vmin=min_attacks, vmax=max_attacks)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, cax=ax, orientation='horizontal')
cb.set_label('Total Regional Air/Drone Attacks (Intensity Scale)')
plt.show()



In [ ]:
# r.to_html('ukraine_3d_map.html')

r.to_py(
)

In [ ]:
code_to_export = """import pandas as pd
import geopandas as gpd
import pydeck as pdk
import matplotlib
import matplotlib.pyplot as plt

# 1. Aggregate Regional Data (Attacks and Fatalities)
# Calculate total fatalities and total attacks for each region (admin1)
regional_stats = acled.groupby('admin1').agg(
    attack_count=('event_id_cnty', 'count'),
    regional_fatalities=('fatalities', 'sum')
).reset_index()

# Normalize the attack count for the color intensity scale
min_attacks = regional_stats['attack_count'].min()
max_attacks = regional_stats['attack_count'].max()
regional_stats['normalized_intensity'] = (regional_stats['attack_count'] - min_attacks) / (max_attacks - min_attacks)

# 2. Load and Prepare Ukraine GeoJSON
geojso_url = "https://raw.githubusercontent.com/martynafford/natural-earth-geojson/master/10m/cultural/ne_10m_admin_1_states_provinces.json"
world_regions = gpd.read_file(geojso_url)

# FIX: Broaden filter to include Crimea/Sevastopol if they are labeled as 'Russia' or 'Disputed'
crimea_names = ['Crimea', 'Avtonomna Respublika Krym', 'Krym', 'Sevastopol', 'm. Sevastopol']
ukraine_regions = world_regions[
    (world_regions['admin'] == 'Ukraine') |
    (world_regions['name'].isin(crimea_names))
].copy()

# Robust Mapping to align GeoJSON names with ACLED
name_mapping = {
    'Kiev': 'Kyiv',
    'Kiev City': 'Kyiv City',
    "Dnipropetrovs'k": 'Dnipropetrovsk',
    'Zaporizhzhya': 'Zaporizhia',
    'Mykolayiv': 'Mykolaiv',
    'Avtonomna Respublika Krym': 'Crimea',
    'Krym': 'Crimea',
    'm. Sevastopol': 'Sevastopol',
    "Donets'k": 'Donetsk',
    'Odessa': 'Odesa',
    "Luhans'k": 'Luhansk',
    "Khmel'nyts'kyy": 'Khmelnytskyi',
    "Ivano-Frankivs'k": 'Ivano-Frankivsk',
    "L'viv": 'Lviv',
    "Ternopil'": 'Ternopil',
    "Vinnytsya": 'Vinnytsia'
}
ukraine_regions['name'] = ukraine_regions['name'].replace(name_mapping)

# Merge aggregated stats into the GeoJSON
ukraine_regions = ukraine_regions.merge(
    regional_stats,
    left_on='name',
    right_on='admin1',
    how='left'
)

# --- Tooltip Preparation: Unify Column Names to avoid "undefined" ---
ukraine_regions['attack_count'] = ukraine_regions['attack_count'].fillna(0)
ukraine_regions['regional_fatalities'] = ukraine_regions['regional_fatalities'].fillna(0)
ukraine_regions['individual_fatalities'] = "N/A (Hover over a bar)"
ukraine_regions['admin1_display'] = ukraine_regions['name']

# Color the regions based on attack intensity
cmap = matplotlib.colormaps['Oranges']
ukraine_regions['fill_color'] = ukraine_regions['normalized_intensity'].apply(
    lambda x: [int(c * 255) for c in cmap(x)[:3]] + [180] if not pd.isna(x) else [0, 0, 0, 0]
)

# 3. Prepare Strike Data (3D Bars)
fatal_strikes = acled[acled['fatalities'] > 0].copy()
fatal_strikes = fatal_strikes.dropna(subset=['longitude', 'latitude'])

# Merge regional stats into individual strike data for the tooltip
fatal_strikes = fatal_strikes.merge(regional_stats, on='admin1', how='left')
fatal_strikes['individual_fatalities'] = fatal_strikes['fatalities']
fatal_strikes['admin1_display'] = fatal_strikes['admin1']

# 4. Define PyDeck Layers
region_layer = pdk.Layer(
    "GeoJsonLayer",
    ukraine_regions,
    opacity=0.8,
    stroked=True,
    filled=True,
    get_fill_color="fill_color",
    get_line_color= [80,80,80],
    pickable=True
)

column_layer = pdk.Layer(
    "ColumnLayer",
    data=fatal_strikes,
    get_position=['longitude', 'latitude'],
    get_elevation='individual_fatalities',
    elevation_scale=3000,
    radius=3000,
    get_fill_color= [136, 8, 8], # Pure Red bars
    pickable=True,
    auto_highlight=True,
)

view_state = pdk.ViewState(latitude=49.0, longitude=32.0, zoom=5.5, pitch=50, bearing=0)

# Unified Tooltip to fix the "undefined" error
tooltip = {
    "html": '''
<b>Region:</b> {admin1_display} <br/>
<b>Total Regional Attacks:</b> {attack_count} <br/>
<b>Total Regional Fatalities:</b> {regional_fatalities} <br/>
<hr style="margin: 5px 0;">
<b>Strike Fatalities:</b> {individual_fatalities}
    ''',
    "style": {"background": "#333333", "color": "white", "font-family": 'Arial', "z-index": "10000"}
}

r = pdk.Deck(layers=[region_layer, column_layer], initial_view_state=view_state, tooltip=tooltip, map_style='dark')

# Display the map
display(r.show())

# 5. Legend for the Poster
fig, ax = plt.subplots(figsize=(8, 1))
fig.subplots_adjust(bottom=0.5)
norm = matplotlib.colors.Normalize(vmin=min_attacks, vmax=max_attacks)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, cax=ax, orientation='horizontal')
cb.set_label('Total Regional Air/Drone Attacks (Intensity Scale)')
plt.show()
"""

with open('ukraine_3d_map.py', 'w') as f:
    f.write(code_to_export)

print("Code exported to 'ukraine_3d_map.py'")

In [ ]:
import streamlit as st
import pandas as pd
import geopandas as gpd
import pydeck as pdk
import matplotlib
import matplotlib.pyplot as plt

# Set up the Streamlit page layout
st.set_page_config(layout="wide", page_title="Ukraine Strikes 3D Map")
st.title("Ukraine Air/Drone Strikes 3D Visualization")
st.markdown("Explore the regional intensity of strikes and specific fatalities. Hover over the regions and 3D bars for details.")

# Cache the data loading so the app runs faster on mobile devices
@st.cache_data
def load_data():
    # Load your ACLED data from GitHub
    acled_url = 'https://raw.githubusercontent.com/lukef533/Ukraine-Missle-Strikes-Data/refs/heads/main/ACLED%20Data_2026-02-13.csv'
    acled = pd.read_csv(acled_url)

    # Load GeoJSON boundaries
    geojson_url = "https://raw.githubusercontent.com/martynafford/natural-earth-geojson/master/10m/cultural/ne_10m_admin_1_states_provinces.json"
    world_regions = gpd.read_file(geojson_url)

    return acled, world_regions

with st.spinner("Loading map data..."):
    acled, world_regions = load_data()

# 1. Aggregate Regional Data (Attacks and Fatalities)
regional_stats = acled.groupby('admin1').agg(
    attack_count=('event_id_cnty', 'count'),
    regional_fatalities=('fatalities', 'sum')
).reset_index()

# Normalize the attack count for the color intensity scale
min_attacks = regional_stats['attack_count'].min()
max_attacks = regional_stats['attack_count'].max()
regional_stats['normalized_intensity'] = (regional_stats['attack_count'] - min_attacks) / (max_attacks - min_attacks)

# 2. Filter and Map GeoJSON
crimea_names = ['Crimea', 'Avtonomna Respublika Krym', 'Krym', 'Sevastopol', 'm. Sevastopol']
ukraine_regions = world_regions[
    (world_regions['admin'] == 'Ukraine') |
    (world_regions['name'].isin(crimea_names))
].copy()

# Robust Mapping to align GeoJSON names with ACLED
name_mapping = {
    'Kiev': 'Kyiv',
    'Kiev City': 'Kyiv City',
    "Dnipropetrovs'k": 'Dnipropetrovsk',
    'Zaporizhzhya': 'Zaporizhia',
    'Mykolayiv': 'Mykolaiv',
    'Avtonomna Respublika Krym': 'Crimea',
    'Krym': 'Crimea',
    'm. Sevastopol': 'Sevastopol',
    "Donets'k": 'Donetsk',
    'Odessa': 'Odesa',
    "Luhans'k": 'Luhansk',
    "Khmel'nyts'kyy": 'Khmelnytskyi',
    "Ivano-Frankivs'k": 'Ivano-Frankivsk',
    "L'viv": 'Lviv',
    "Ternopil'": 'Ternopil',
    "Vinnytsya": 'Vinnytsia'
}
ukraine_regions['name'] = ukraine_regions['name'].replace(name_mapping)

# Merge aggregated stats into the GeoJSON
ukraine_regions = ukraine_regions.merge(
    regional_stats,
    left_on='name',
    right_on='admin1',
    how='left'
)

# Tooltip Preparation: Unify Column Names
ukraine_regions['attack_count'] = ukraine_regions['attack_count'].fillna(0)
ukraine_regions['regional_fatalities'] = ukraine_regions['regional_fatalities'].fillna(0)
ukraine_regions['individual_fatalities'] = "N/A (Hover over a bar)"
ukraine_regions['admin1_display'] = ukraine_regions['name']

# Color the regions based on attack intensity
cmap = matplotlib.colormaps['Oranges']
ukraine_regions['fill_color'] = ukraine_regions['normalized_intensity'].apply(
    lambda x: [int(c * 255) for c in cmap(x)[:3]] + [180] if not pd.isna(x) else [0, 0, 0, 0]
)

# 3. Prepare Strike Data (3D Bars)
fatal_strikes = acled[acled['fatalities'] > 0].copy()
fatal_strikes = fatal_strikes.dropna(subset=['longitude', 'latitude'])

# Merge regional stats into individual strike data for the tooltip
fatal_strikes = fatal_strikes.merge(regional_stats, on='admin1', how='left')
fatal_strikes['individual_fatalities'] = fatal_strikes['fatalities']
fatal_strikes['admin1_display'] = fatal_strikes['admin1']

# 4. Define PyDeck Layers
region_layer = pdk.Layer(
    "GeoJsonLayer",
    ukraine_regions,
    opacity=0.8,
    stroked=True,
    filled=True,
    get_fill_color="fill_color",
    get_line_color= [80,80,80],
    pickable=True
)

column_layer = pdk.Layer(
    "ColumnLayer",
    data=fatal_strikes,
    get_position=['longitude', 'latitude'],
    get_elevation='individual_fatalities',
    elevation_scale=3000,
    radius=3000,
    get_fill_color= [136, 8, 8], # Pure Red bars
    pickable=True,
    auto_highlight=True,
)

view_state = pdk.ViewState(latitude=49.0, longitude=32.0, zoom=5.2, pitch=50, bearing=0)

tooltip = {
    "html": """
        <b>Region:</b> {admin1_display} <br/>
        <b>Total Regional Attacks:</b> {attack_count} <br/>
        <b>Total Regional Fatalities:</b> {regional_fatalities} <br/>
        <hr style="margin: 5px 0;">
        <b>Strike Fatalities:</b> {individual_fatalities}
    """,
    "style": {"background": "#333333", "color": "white", "font-family": 'Arial', "z-index": "10000"}
}

r = pdk.Deck(layers=[region_layer, column_layer], initial_view_state=view_state, tooltip=tooltip, map_style='dark')

# Render the Map in Streamlit
st.pydeck_chart(r)

# 5. Render the Legend in Streamlit
fig, ax = plt.subplots(figsize=(8, 1))
fig.subplots_adjust(bottom=0.5)
norm = matplotlib.colors.Normalize(vmin=min_attacks, vmax=max_attacks)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, cax=ax, orientation='horizontal')
cb.set_label('Total Regional Air/Drone Attacks (Intensity Scale)')

# Render the Matplotlib figure
st.pyplot(fig)

In [ ]:
# [80,80,80]
# [255,255,255]
# [136, 8, 8]
# name_mapping = {
#     'Kiev': 'Kyiv',
#     'Kiev City': 'Kyiv City',
#     "Dnipropetrovs'k": 'Dnipropetrovsk',
#     'Zaporizhzhya': 'Zaporizhia',
#     'Mykolayiv': 'Mykolaiv',
#     "Donets'k": 'Donetsk',
#     'Odessa': 'Odesa',
#     "Luhans'k": 'Luhansk',
#     "Khmel'nyts'kyy": 'Khmelnytskyi',
#     "Ivano-Frankivs'k": 'Ivano-Frankivsk',
#     "L'viv": 'Lviv',
#     "Ternopil'": 'Ternopil'
# }

# Modeling

## Prepping Acled Dataset for Modeling

In [ ]:
acled_model_df = acled.copy()

cols_to_drop = ['iso', 'disorder_type', 'event_type', 'sub_event_type', 'region', 'country', 'source', 'source_scale', 'notes', 'timestamp', 'tags', 'assoc_actor_2', 'assoc_actor_1']
acled_model_df = acled_model_df.drop(columns=cols_to_drop, errors='ignore')

# Convert civilian_targeting to a numerical (dummy) variable
acled_model_df['civilian_targeting'] = acled_model_df['civilian_targeting'].apply(lambda x: 1 if x == 'Civilian targeting' else 0)

# fill nulls in population best with 0, only 2 values
acled_model_df['population_best'] = acled_model_df['population_best'].fillna(0)

# === NEW FEATURE ENGINEERING: Temporal Features from event_date ===
# Ensure event_date is datetime
acled_model_df['event_date'] = pd.to_datetime(acled_model_df['event_date'])

# Extract temporal features
acled_model_df['event_month'] = acled_model_df['event_date'].dt.month
acled_model_df['event_day_of_week'] = acled_model_df['event_date'].dt.dayofweek
acled_model_df['event_day_of_year'] = acled_model_df['event_date'].dt.dayofyear

# drop the rest of the null rows and display final shape
acled_model_df = acled_model_df.dropna()
print('rows removed: ' + str(acled.shape[0] - acled_model_df.shape[0]))
print('cols removed: ' + str(acled.shape[1] - acled_model_df.shape[1]))
print(acled_model_df.shape)
print(acled_model_df.isnull().sum())

## Geospatial Feature Engineering: K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Prepare data for clustering (scale latitude and longitude)
coords = acled_model_df[['latitude', 'longitude']].copy()
scaler = MinMaxScaler()
coords_scaled = scaler.fit_transform(coords)

# --- Determine optimal K using the Elbow Method ---
wcss = [] # Within-cluster sum of squares
for i in range(1, 11): # Test k from 1 to 10
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init='auto')
    kmeans.fit(coords_scaled)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method results
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method to Determine Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.grid(True)
plt.show()

print("Review the plot above to choose an optimal K. A common choice is where the 'elbow' in the plot occurs.")
print("For now, we'll proceed with K=5 as per your request.")

# Apply K-Means clustering with a chosen K (e.g., K=5)
optimal_k = 5 # Based on user request
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init='auto')
acled_model_df['geospatial_cluster'] = kmeans.fit_predict(coords_scaled)

print(f"Added 'geospatial_cluster' feature with {optimal_k} clusters.")
print(acled_model_df[['latitude', 'longitude', 'geospatial_cluster']].head())

### Visualization of Geospatial Clusters

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(x='longitude', y='latitude', hue='geospatial_cluster', data=acled_model_df, palette='viridis', s=20, alpha=0.7)
plt.title('Geospatial Clusters of ACLED Events')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## Lasso Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Define target variable
y = acled_model_df['fatalities']

# Select numerical features, including temporal and geospatial features
X_numeric_cols = ['year', 'time_precision', 'latitude', 'longitude', 'geo_precision',
                  'population_best', 'civilian_targeting',
                  'event_month', 'event_day_of_week', 'event_day_of_year',
                  'geospatial_cluster'] # Added 'geospatial_cluster'
X_numeric = acled_model_df[X_numeric_cols]

# One-hot encode the 'interaction' categorical feature
X_categorical = pd.get_dummies(acled_model_df['interaction'], prefix='interaction')

# Combine numerical and one-hot encoded categorical features
X = pd.concat([X_numeric, X_categorical], axis=1)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features (important for Lasso regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Initialize and train the Lasso model
# Alpha is the regularization strength. A value of 0 means no regularization (equivalent to Linear Regression)
lasso_model = Lasso(alpha=0.1, random_state=42)
lasso_model.fit(X_train_scaled, y_train)

print("Lasso Model re-trained successfully with updated features!")

In [ ]:
# Make predictions on the test set
y_pred = lasso_model.predict(X_test_scaled)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

# Display coefficients
lasso_coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': lasso_model.coef_})
print(f"\nLasso Coefficients (K={optimal_k}):")
display(lasso_coefficients.sort_values(by='Coefficient', ascending=False))

## Analyze Target Variable Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(acled_model_df['fatalities'], bins=50, kde=True)
plt.title('Distribution of Fatalities')
plt.xlabel('Fatalities')
plt.ylabel('Frequency')
plt.xlim(0, acled_model_df['fatalities'].quantile(0.99)) # Limit x-axis to 99th percentile for better visualization of main distribution
plt.grid(True)
plt.show()

print(f"Fatalities describe:\n{acled_model_df['fatalities'].describe()}")
print(f"\nNumber of events with 0 fatalities: {(acled_model_df['fatalities'] == 0).sum()} ({(acled_model_df['fatalities'] == 0).mean()*100:.2f}%) ")

## Binary Classification (Fatalities > 0)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create the binary target variable
acled_model_df['fatalities_binary'] = (acled_model_df['fatalities'] > 0).astype(int)

# Define target variable for binary classification
y_binary = acled_model_df['fatalities_binary']

# --- Ensure X and related feature definitions are available ---
# Define target variable for regression (re-using the logic from Lasso setup)
y = acled_model_df['fatalities']

# Select numerical features, including temporal and geospatial features
X_numeric_cols = ['year', 'time_precision', 'latitude', 'longitude', 'geo_precision',
                  'population_best', 'civilian_targeting',
                  'event_month', 'event_day_of_week', 'event_day_of_year',
                  'geospatial_cluster']
X_numeric = acled_model_df[X_numeric_cols]

# One-hot encode the 'interaction' categorical feature
X_categorical = pd.get_dummies(acled_model_df['interaction'], prefix='interaction')

# Combine numerical and one-hot encoded categorical features
X = pd.concat([X_numeric, X_categorical], axis=1)
# --- End of X and related definitions ---

# Split data into training and testing sets for binary classification
X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(X, y_binary, test_size=0.2, random_state=42, stratify=y_binary) # Stratify to maintain class balance

# Scale the features
scaler_binary = StandardScaler()
X_train_scaled_binary = scaler_binary.fit_transform(X_train_binary)
X_test_scaled_binary = scaler_binary.transform(X_test_binary)

print(f"X_train_binary shape: {X_train_binary.shape}")
print(f"y_train_binary shape: {y_train_binary.shape}")
print(f"Class distribution in y_train_binary:\n{y_train_binary.value_counts(normalize=True)}")
print(f"Class distribution in y_test_binary:\n{y_test_binary.value_counts(normalize=True)}")

### Logistic Regression for Binary Classification

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced') # Use 'liblinear' for small datasets, 'balanced' for imbalanced classes
log_reg_model.fit(X_train_scaled_binary, y_train_binary)

print("Logistic Regression model trained successfully!")

In [ ]:
# Make predictions on the test set
y_pred_binary = log_reg_model.predict(X_test_scaled_binary)
y_pred_proba_binary = log_reg_model.predict_proba(X_test_scaled_binary)[:, 1] # Probability of class 1

# Evaluate the model
accuracy = accuracy_score(y_test_binary, y_pred_binary)
precision = precision_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")

# Confusion Matrix
cm = confusion_matrix(y_test_binary, y_pred_binary)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=log_reg_model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Logistic Regression')
plt.show()

# Display coefficients
log_reg_coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': log_reg_model.coef_[0]})
print("\nLogistic Regression Coefficients:")
display(log_reg_coefficients.sort_values(by='Coefficient', ascending=False))

### Random Forest Classifier for Binary Classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest Classifier model
# Using default parameters for initial run, can be tuned later
rf_classifier_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced') # n_jobs=-1 uses all available cores
rf_classifier_model.fit(X_train_binary, y_train_binary)

print("Random Forest Classifier trained successfully!")

In [ ]:
# Make predictions on the test set
y_pred_binary_rf = rf_classifier_model.predict(X_test_binary)
y_pred_proba_binary_rf = rf_classifier_model.predict_proba(X_test_binary)[:, 1] # Probability of class 1

# Evaluate the model
accuracy_rf = accuracy_score(y_test_binary, y_pred_binary_rf)
precision_rf = precision_score(y_test_binary, y_pred_binary_rf)
recall_rf = recall_score(y_test_binary, y_pred_binary_rf)
f1_rf = f1_score(y_test_binary, y_pred_binary_rf)

print(f"Random Forest Classifier Accuracy: {accuracy_rf:.2f}")
print(f"Random Forest Classifier Precision: {precision_rf:.2f}")
print(f"Random Forest Classifier Recall: {recall_rf:.2f}")
print(f"Random Forest Classifier F1-Score: {f1_rf:.2f}")

# Confusion Matrix
cm_rf = confusion_matrix(y_test_binary, y_pred_binary_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=rf_classifier_model.classes_)
disp_rf.plot(cmap=plt.cm.Greens)
plt.title('Confusion Matrix for Random Forest Classifier')
plt.show()

# Display feature importances
feature_importances_rf = pd.DataFrame({'Feature': X.columns, 'Importance': rf_classifier_model.feature_importances_})
print("\nRandom Forest Classifier Feature Importances:")
display(feature_importances_rf.sort_values(by='Importance', ascending=False).head(10))

## Poster Text Summaries

### 1. Project Overview & Objective

**Understanding the Conflict in Ukraine**
This project analyzes the Ukraine conflict using two key datasets: ACLED (Armed Conflict Location & Event Data Project) and detailed missile/UAV strike data. Our goal is to uncover patterns in conflict events, missile attacks, and their impact, particularly on civilian fatalities.

### 2. Data Sources

**Comprehensive Conflict Data**
We utilized:
*   **ACLED Data**: Provides granular information on conflict events, actors, locations, and fatalities across Ukraine.
*   **Missile/UAV Attack Data**: Detailed records of launched and destroyed missiles and unmanned aerial vehicles, including their models and targets.

### 3. Analytical Methods

**Exploring Patterns & Predicting Impact**
Our approach involved:
*   **Exploratory Data Analysis (EDA)**: Visualizing event types, timelines, and regional impact to understand general trends.
*   **Geospatial Analysis**: Mapping conflict hotspots and fatality concentrations, including a 3D visualization of regional attack intensity and event-level fatalities.
*   **Feature Engineering**: Creating new variables like temporal features (month, day of week) and geospatial clusters to enhance predictive power.
*   **Predictive Modeling**: Employing Lasso Regression and Random Forest models to identify factors influencing conflict fatalities. A specialized two-part model was developed to first predict if fatalities occur, then to estimate their number when they do.

### 4. Key Findings & Results

**Insights from the Conflict**
Our investigation in the Ukraine conflict, with a special focus on UAV strikes, has uncovered significant insights. The conflict involves a substantial number of events, with over 6,000 recorded incidents resulting in more than 10,000 fatalities. Geographically, intense conflict activity is concentrated in specific regions, leading to a disproportionate impact on those areas and a higher likelihood of casualties. Our analysis of missile and UAV data also reveals an overall destruction rate of over 70% for launched projectiles, indicating strong defensive capabilities. However, a notable proportion of conflict events, specifically over 50%, are recorded without any fatalities, highlighting the varied nature and impact of different incidents. Our predictive analysis further confirms that **where and when these events occur** are crucial factors in determining their severity and the number of casualties. This predictive capability offers a powerful tool for understanding and potentially mitigating the human cost of these attacks.

### 5. Broader Implications & Future Work

**Understanding Conflict for Better Responses**
*   **Humanitarian Aid**: Identifying high-risk areas and understanding casualty patterns can inform targeted humanitarian efforts.
*   **Policy & Defense**: Insights into attack types and locations can aid in developing more effective defense strategies and resource allocation.
*   **Conflict Monitoring**: Predictive models can contribute to early warning systems for escalating violence.

**Future Directions**:
Future work could integrate more diverse data sources (e.g., economic indicators, social media sentiment), explore advanced deep learning models for spatial-temporal predictions, and further refine models to predict specific types of casualties or damages.

### 6. Key Statistics (Big Ass Numbers for your Poster!)

In [ ]:
# Calculate and display 'Big Ass Numbers' (BANs)

# 1. Total ACLED Events (already calculated in EDA)
total_acled_events = len(acled)
print(f"Total recorded conflict events (ACLED): {total_acled_events}")

# 2. Total Fatalities (already calculated in EDA)
total_fatalities = acled['fatalities'].sum()
print(f"Total recorded fatalities from conflict events (ACLED): {total_fatalities}")

# 3. Overall Missile/UAV Destruction Rate (already calculated in EDA)
total_launched = missile_combined['launched'].sum()
total_destroyed = missile_combined['destroyed'].sum()
overall_destruction_rate = (total_destroyed / total_launched) * 100 if total_launched > 0 else 0
print(f"Overall Missile/UAV Destruction Rate: {overall_destruction_rate:.1f}%")

# 4. Percentage of events with zero fatalities (already calculated in 'Analyze Target Variable Distribution')
zero_fatalities_percentage = (acled_model_df['fatalities'] == 0).mean() * 100
print(f"Percentage of conflict events with NO recorded fatalities: {zero_fatalities_percentage:.1f}%")

# 5. Top affected region (already calculated in EDA)
top_region = acled['admin1'].value_counts().index[0]
top_region_count = acled['admin1'].value_counts().iloc[0]
print(f"Most affected region: {top_region} with {top_region_count} events.")

# 6. Number of unique locations in ACLED data
unique_acled_locations = acled['location'].nunique()
print(f"Number of unique conflict locations recorded (ACLED): {unique_acled_locations}")

In [ ]:
print(combined_uav_only.target_main.unique())
print(combined_uav_only.target_main.isnull().sum())
print(combined_uav_only.shape)

In [ ]:
total_launched
missile_combined.time_start.max()

In [ ]:
uav_launches_by_model = missile_combined[missile_combined['category'] == 'UAV'].groupby('model')['launched'].sum().sort_values(ascending=False)
print("Total UAVs Launched by Model:")
display(uav_launches_by_model)

In [ ]:
donetsk_luhansk_strikes = acled[acled['admin1'].isin(['Donetsk', 'Luhansk'])]
num_donetsk_luhansk_strikes = len(donetsk_luhansk_strikes)
total_acled_strikes = len(acled)

percentage_donetsk_luhansk = (num_donetsk_luhansk_strikes / total_acled_strikes) * 100

print(f"Number of strikes in Donetsk and Luhansk: {num_donetsk_luhansk_strikes}")
print(f"Percentage of total strikes in Donetsk and Luhansk: {percentage_donetsk_luhansk:.2f}%")

In [ ]:
shahed_strikes = missile_combined[missile_combined['model'] == 'Shahed-136/131']['launched'].sum()
total_missile_strikes = missile_combined['launched'].sum()

percentage_shahed = (shahed_strikes / total_missile_strikes) * 100

print(f"Total strikes by Shahed-136/131: {shahed_strikes:.0f}")
print(f"Percentage of total strikes by Shahed-136/131: {percentage_shahed:.2f}%")